In [135]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd


In [136]:
# uncomment this if you want to download the data directly
# raw_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
# orders = pd.read_csv(raw_dir / "olist_orders_dataset.csv")

raw_dir = Path("../data/raw")

orders = pd.read_csv(raw_dir / "olist_orders_dataset.csv")
customers = pd.read_csv(raw_dir / "olist_customers_dataset.csv")
geolocation = pd.read_csv(raw_dir / "olist_geolocation_dataset.csv")
items = pd.read_csv(raw_dir / "olist_order_items_dataset.csv")
payments = pd.read_csv(raw_dir / "olist_order_payments_dataset.csv")
products = pd.read_csv(raw_dir / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_dir / "olist_sellers_dataset.csv")
category_translation= pd.read_csv(raw_dir / "product_category_name_translation.csv")

In [137]:
print(orders.columns)
print(60*"*")
print(customers.columns)
print(60*"*")
print(geolocation.columns)
print(60*"*")
print(items.columns)
print(60*"*")
print(payments.columns)
print(60*"*")
print(products.columns)
print(60*"*")
print(sellers.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
************************************************************
Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')
************************************************************
Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str')
************************************************************
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')
************************************************************
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
  

In [138]:
print("orders shape:", orders.shape)
print("customers shape:", customers.shape)
print(60*"*")
print("NaN in order data:\n",orders.isna().sum())
print(60*"*")
print("NaN in order data:\n",customers.isna().sum())


orders shape: (99441, 8)
customers shape: (99441, 5)
************************************************************
NaN in order data:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
************************************************************
NaN in order data:
 customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


## Extracting and Analysing the Target Variable
The target is delivary delays, so when the difference between the estimated and actual delivery date is negative, it means that the delay is happend, and the target value will be 1, otherwise 0.  

In [139]:
target = pd.to_datetime(orders["order_estimated_delivery_date"]) - pd.to_datetime(orders["order_delivered_customer_date"]) < pd.Timedelta(0)
target = (target == 1).astype(int)
target.value_counts()

0    91614
1     7827
Name: count, dtype: int64

In [140]:
# Removing the target variable to prevent the leakage 
# and features that are not available at the time of inference (at the moment of purchase)
del orders["order_delivered_customer_date"]
del orders["order_status"]
del orders["order_delivered_carrier_date"]

## Extracting Features for the baseline model

We want to our data have rows for each order_id, so all other dataframes should be reduced to one row per order_id to be able to merge all together. 

### Merging the orders and customers tables

In [141]:
df_merged = pd.merge(orders, customers, on="customer_id", how="left")
# print(df_merged.isna().sum())
# df_merged.head()
print(df_merged.shape)
print(df_merged.columns)


(99441, 9)
Index(['order_id', 'customer_id', 'order_purchase_timestamp',
       'order_approved_at', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state'],
      dtype='str')


### Extracting features from "items" table
 - item counts
 - unique products counts 
 - seller counts
 - total price of the order
 - total price of the freight 

In [142]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [143]:
item_counts = items.groupby("order_id").size().rename("item_count")
product_counts = (
    items.groupby("order_id")["product_id"]
    .nunique()
    .rename("distinct_product_count")
)
seller_counts = (
    items.groupby("order_id")["seller_id"]
    .nunique()
    .rename("seller_count")
)

total_order_price = items.groupby("order_id")["price"].sum().rename("total_order_price")
total_freight_value = items.groupby("order_id")["freight_value"].sum().rename("total_freight_value")

item_features = pd.concat([item_counts, product_counts, seller_counts, total_order_price, total_freight_value], axis=1).reset_index()

item_features.head()


,order_id,item_count,distinct_product_count,seller_count,total_order_price,total_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


In [144]:
item_features.isna().sum()

order_id                  0
item_count                0
distinct_product_count    0
seller_count              0
total_order_price         0
total_freight_value       0
dtype: int64

In [145]:
df_merged = df_merged.merge(item_features, on="order_id", how="left", validate="one_to_one")

In [146]:
print(df_merged.shape)
print(df_merged.isna().sum())
# df_merged[df_merged.isnull().any(axis=1)]

(99441, 14)
order_id                           0
customer_id                        0
order_purchase_timestamp           0
order_approved_at                160
order_estimated_delivery_date      0
customer_unique_id                 0
customer_zip_code_prefix           0
customer_city                      0
customer_state                     0
item_count                       775
distinct_product_count           775
seller_count                     775
total_order_price                775
total_freight_value              775
dtype: int64


### Extracting features from payment tables